# 02 — Saída estruturada com Pydantic

**Trilha:** [01 chamada e JSON](01_chamada_e_json.ipynb) → **02 saída estruturada** → [03 enums e guardrails](03_enums_e_guardrails.ipynb)

**De onde viemos:** no [01](01_chamada_e_json.ipynb), o JSON pedido no prompt veio com a quantidade como texto (`"250 grams"`),
e nada garantia o formato. Aqui a lista de compras ganha um **contrato**: uma classe Pydantic que diz quais campos existem e de
que tipo — e que confere a resposta.

## 1. O contrato

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # prettifier.py e logger_general.py ficam em AI_use_cases/
from prettifier import show, show_panel, show_table

In [2]:
import json
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field, ValidationError


class Ingrediente(BaseModel):
    model_config = ConfigDict(extra="forbid")
    nome: str
    quantidade: float = Field(gt=0, description="Quantidade numérica, sem unidade")
    unidade: str = Field(description="Unidade métrica: g, kg, ml, l ou un")


class ListaDeCompras(BaseModel):
    model_config = ConfigDict(extra="forbid")
    prato: str
    porcoes: int = Field(gt=0)
    ingredientes: list[Ingrediente]

Duas mudanças em relação ao 01: a quantidade é **número** e a unidade é um campo **separado**. O Pydantic gera, a partir da
classe, o JSON Schema que descreve esse contrato — é isso que vai para o modelo:

In [3]:
schema = ListaDeCompras.model_json_schema()
show(f"campos de ListaDeCompras: {list(schema['properties'])}")
show(f"campos de Ingrediente: {list(schema['$defs']['Ingrediente']['properties'])}")
show(f"quantidade: {schema['$defs']['Ingrediente']['properties']['quantidade']}")

campos de ListaDeCompras: ['prato', 'porcoes', 'ingredientes']
campos de Ingrediente: ['nome', 'quantidade', 'unidade']
quantidade: {'description': 'Quantidade numérica, sem unidade', 'exclusiveMinimum': 0, 'title': 'Quantidade', 
'type': 'number'}


Configuração do cliente, igual ao 01 (célula de API: não roda sem `.env`):

In [4]:
# api: configuração do cliente — lê a chave e o endpoint do .env
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.getenv("openai")
if not api_key:
    raise ValueError("variável 'openai' (chave da API) não encontrada no .env")
openai_base_url = os.getenv("openai_base_url")
if not openai_base_url:
    raise ValueError("variável 'openai_base_url' não encontrada no .env")

AI_MODEL = "gpt-4o-mini"
ai_client = OpenAI(api_key=api_key, base_url=openai_base_url)

## 2. Três jeitos de pedir a resposta no formato do contrato

### 2.1 Schema no prompt + validação local (funciona em qualquer endpoint)

In [4]:
def prompt_com_schema(modelo: type[BaseModel]) -> str:
    return (
        "Você gera listas de compras. Use ingredientes in natura sempre que possível e o sistema métrico.\n"
        "Responda SOMENTE com um JSON válido que siga este JSON Schema:\n"
        f"{json.dumps(modelo.model_json_schema(), ensure_ascii=False)}"
    )


show_panel(prompt_com_schema(ListaDeCompras), title="prompt_com_schema(ListaDeCompras)")

╭─ prompt_com_schema(ListaDeCompras) ────────────────────────────────────────────────────────────────────────╮
│ Você gera listas de compras. Use ingredientes in natura sempre que possível e o sistema métrico.           │
│ Responda SOMENTE com um JSON válido que siga este JSON Schema:                                             │
│ {                                                                                                          │
│   "$defs": {                                                                                               │
│     "Ingrediente": {                                                                                       │
│       "additionalProperties": false,                                                                       │
│       "properties": {                                                                                      │
│         "nome": {                                                                                          │
│

In [6]:
# api: schema no prompt, validação com Pydantic
chat_completion = ai_client.chat.completions.create(
    model=AI_MODEL,
    messages=[
        {"role": "system", "content": prompt_com_schema(ListaDeCompras)},
        {"role": "user", "content": "Pizza margherita para 2 pessoas"},
    ],
)
resposta_ao_vivo = chat_completion.choices[0].message.content
lista_ao_vivo = ListaDeCompras.model_validate_json(resposta_ao_vivo)
show(f"lista_ao_vivo: {lista_ao_vivo}")

lista_ao_vivo: prato='Pizza Margherita' porcoes=2 ingredientes=[Ingrediente(nome='farinha de trigo', 
quantidade=250.0, unidade='g'), Ingrediente(nome='água', quantidade=150.0, unidade='ml'), 
Ingrediente(nome='fermento biológico', quantidade=5.0, unidade='g'), Ingrediente(nome='sal', quantidade=5.0, 
unidade='g'), Ingrediente(nome='azeite de oliva', quantidade=10.0, unidade='ml'), Ingrediente(nome='molho de 
tomate', quantidade=100.0, unidade='g'), Ingrediente(nome='queijo mozzarella', quantidade=150.0, unidade='g'),
Ingrediente(nome='manjericão fresco', quantidade=10.0, unidade='g')]


### 2.2 Saída estruturada nativa (`.parse`) — se o endpoint suportar

Alguns provedores aceitam o modelo Pydantic direto e **restringem a geração** ao schema. O SDK valida e devolve o objeto pronto
em `.parsed`. Nem todo endpoint compatível com a OpenAI suporta isso; se o seu não suportar, use o 2.1.

In [7]:
# api: saída estruturada nativa
completion = ai_client.chat.completions.parse(
    model=AI_MODEL,
    messages=[
        {"role": "system", "content": "Você gera listas de compras com ingredientes in natura e sistema métrico."},
        {"role": "user", "content": "Pizza margherita para 2 pessoas"},
    ],
    response_format=ListaDeCompras,
)
lista_parse = completion.choices[0].message.parsed
show(f"lista_parse: {lista_parse}")

lista_parse: prato='Pizza Margherita' porcoes=2 ingredientes=[Ingrediente(nome='farinha de trigo', 
quantidade=250.0, unidade='g'), Ingrediente(nome='água', quantidade=150.0, unidade='ml'), 
Ingrediente(nome='fermento biológico', quantidade=5.0, unidade='g'), Ingrediente(nome='sal', quantidade=5.0, 
unidade='g'), Ingrediente(nome='queijo mozzarella', quantidade=150.0, unidade='g'), Ingrediente(nome='molho de
tomate', quantidade=100.0, unidade='ml'), Ingrediente(nome='manjericão fresco', quantidade=5.0, unidade='un'),
Ingrediente(nome='azeite de oliva', quantidade=10.0, unidade='ml')]


### 2.3 Com pydantic-ai (o `Agent` do seu notebook)

O `Agent` faz o mesmo com menos código: recebe `output_type=ListaDeCompras` e devolve o objeto validado. (O `OpenAIModel` do
exemplo original foi renomeado para `OpenAIChatModel`; o endpoint entra pelo `OpenAIProvider`.)

> **Pegadinha:** dentro do Jupyter use `await agente.run(...)`. O `agente.run_sync(...)` tenta abrir um event loop, e o kernel
> já tem um rodando: `RuntimeError: This event loop is already running`. Em script `.py` comum, `run_sync` funciona.

In [8]:
# api: pydantic-ai com o mesmo endpoint
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

modelo_pai = OpenAIChatModel(AI_MODEL, provider=OpenAIProvider(base_url=openai_base_url, api_key=api_key))
agente_compras = Agent(modelo_pai, output_type=ListaDeCompras,
                       instructions="Você gera listas de compras com ingredientes in natura e sistema métrico.")
resultado = await agente_compras.run("Pizza margherita para 2 pessoas")  # no Jupyter: await, não run_sync
show(f"resultado.output: {resultado.output}")

resultado.output: prato='Pizza Margherita' porcoes=2 ingredientes=[Ingrediente(nome='Farinha de trigo', 
quantidade=200.0, unidade='g'), Ingrediente(nome='Água', quantidade=120.0, unidade='ml'), 
Ingrediente(nome='Fermento fresco', quantidade=5.0, unidade='g'), Ingrediente(nome='Sal', quantidade=5.0, 
unidade='g'), Ingrediente(nome='Azeite de oliva', quantidade=10.0, unidade='ml'), Ingrediente(nome='Molho de 
tomate', quantidade=100.0, unidade='g'), Ingrediente(nome='Queijo mozzarella', quantidade=150.0, unidade='g'),
Ingrediente(nome='Manjericão fresco', quantidade=5.0, unidade='g')]


Sem chave, dá para ver o `Agent` funcionando com o `TestModel` do pydantic-ai, um modelo **falso** que gera dados que só
obedecem ao schema (os valores não fazem sentido — servem para testar o encanamento):

In [5]:
import os

os.environ["PYDANTIC_AI_NO_BANNER"] = "1"
from pydantic_ai import Agent
from pydantic_ai.models.test import TestModel

agente_teste = Agent(TestModel(), output_type=ListaDeCompras)
saida_teste = (await agente_teste.run("Pizza margherita para 2 pessoas")).output
show(f"type(saida_teste).__name__: {type(saida_teste).__name__}")
show(f"saida_teste: {saida_teste!r}")

type(saida_teste).__name__: ListaDeCompras
saida_teste: ListaDeCompras(prato='a', porcoes=1, ingredientes=[Ingrediente(nome='a', quantidade=1.0, 
unidade='a')])


## 3. Validando respostas

A resposta **real** gravada no 01 seguia o formato pedido lá: uma **lista** solta de `{"ingredients", "amount"}`, sem prato nem
porções. Contra o contrato novo, ela é recusada logo no primeiro nível:

In [6]:
resposta_antiga = Path("../respostas_gravadas/pizza_margherita_real.json").read_text(encoding="utf-8")
try:
    ListaDeCompras.model_validate_json(resposta_antiga)
except ValidationError as ex:
    show(f"erros: {ex.error_count()}")
    show(f"primeiro: {ex.errors()[0]['msg']}")

primeiro_item = json.loads(resposta_antiga)[0]
try:
    Ingrediente.model_validate(primeiro_item)
except ValidationError as ex:
    for erro in ex.errors():
        show(f"{erro['loc']}: {erro['msg']}")

erros: 1
primeiro: Input should be an object
('nome',): Field required
('quantidade',): Field required
('unidade',): Field required
('ingredients',): Extra inputs are not permitted
('amount',): Extra inputs are not permitted


É por isso que o contrato **precisa ir para o modelo** (seções 2.1 a 2.3): o Pydantic confere, mas quem gera é o modelo.

Uma resposta no formato novo (*exemplo construído*, com as quantidades da resposta real):

In [7]:
resposta_nova = """
{"prato": "Pizza margherita", "porcoes": 2, "ingredientes": [
  {"nome": "farinha de trigo", "quantidade": 250, "unidade": "g"},
  {"nome": "fermento biológico seco", "quantidade": 5, "unidade": "g"},
  {"nome": "água", "quantidade": 150, "unidade": "ml"},
  {"nome": "azeite", "quantidade": 15, "unidade": "ml"},
  {"nome": "molho de tomate", "quantidade": 100, "unidade": "g"},
  {"nome": "muçarela de búfala", "quantidade": 150, "unidade": "g"},
  {"nome": "manjericão fresco", "quantidade": 10, "unidade": "g"}
]}
"""
lista = ListaDeCompras.model_validate_json(resposta_nova)
show(f"lista.prato: {lista.prato}")
show(f"lista.porcoes: {lista.porcoes}")
show(f"lista.ingredientes[0]: {lista.ingredientes[0]!r}")

lista.prato: Pizza margherita
lista.porcoes: 2
lista.ingredientes[0]: Ingrediente(nome='farinha de trigo', quantidade=250.0, unidade='g')


Agora o problema do 01 some: a quantidade é número.

In [8]:
def para_porcoes(lista: ListaDeCompras, porcoes: int) -> ListaDeCompras:
    fator = porcoes / lista.porcoes
    return ListaDeCompras(
        prato=lista.prato,
        porcoes=porcoes,
        ingredientes=[item.model_copy(update={"quantidade": item.quantidade * fator}) for item in lista.ingredientes],
    )


para_quatro = para_porcoes(lista, 4)
show_table(["ingrediente", "2 porções", "4 porções"],
       [(antes.nome, f"{antes.quantidade:g} {antes.unidade}", f"{depois.quantidade:g} {depois.unidade}")
        for antes, depois in zip(lista.ingredientes, para_quatro.ingredientes)])

                                                   
  ingrediente               2 porções   4 porções  
 ───────────────────────────────────────────────── 
  farinha de trigo          250 g       500 g      
  fermento biológico seco   5 g         10 g       
  água                      150 ml      300 ml     
  azeite                    15 ml       30 ml      
  molho de tomate           100 g       200 g      
  muçarela de búfala        150 g       300 g      
  manjericão fresco         10 g        20 g       
                                                   


## ⚠️ O que ainda quebra (e o [03](03_enums_e_guardrails.ipynb) resolve)

**1. "Unidade métrica" está só na descrição.** O campo `unidade` é `str`: qualquer texto passa (*exemplo construído*):

In [9]:
imperial = """
{"prato": "Pizza margherita", "porcoes": 2, "ingredientes": [
  {"nome": "flour", "quantidade": 2, "unidade": "cups"},
  {"nome": "olive oil", "quantidade": 1, "unidade": "tablespoon"}
]}
"""
aceita = ListaDeCompras.model_validate_json(imperial)
show(f"unidades aceitas: {[item.unidade for item in aceita.ingredientes]}")

unidades aceitas: ['cups', 'tablespoon']


O `description` do campo é um **pedido** ao modelo, não uma regra. A validação passou e a lista está em xícaras.

**2. Texto em volta do JSON quebra a seção 2.1, e recortar texto não resolve.** O `model_validate_json` exige JSON puro,
e a resposta real do 01 veio dentro de uma cerca ```` ```json ````:

In [10]:
com_cerca = 'Aqui está:\n```json\n' + resposta_nova.strip() + '\n```'
try:
    ListaDeCompras.model_validate_json(com_cerca)
except ValidationError as ex:
    show(f"com cerca: {ex.errors()[0]['type']}")

com cerca: json_invalid


A saída óbvia é **recortar o texto** da primeira `{` até a última `}`. Funciona com a cerca, mas falha em respostas comuns
(*exemplos construídos*) — e numa delas falha **em silêncio**:

In [11]:
def recortar_json(texto: str) -> str:
    """Recorte por texto, da primeira '{' até a última '}'. Frágil: está aqui só para comparação."""
    inicio, fim = texto.find("{"), texto.rfind("}")
    if inicio == -1 or fim == -1:
        raise ValueError("nenhum objeto JSON na resposta")
    return texto[inicio:fim + 1]


item = '{"nome": "sal", "quantidade": 5, "unidade": "g"}'
respostas = {
    "cerca ```json": com_cerca,
    "texto depois com chaves": resposta_nova.strip() + "\nObs: troque {queijo} se quiser.",
    "texto antes com chaves": "Formato {prato, porcoes, ingredientes}:\n" + resposta_nova.strip(),
    "lista solta": "[" + item + "]",
}
for nome, texto in respostas.items():
    try:
        dado = json.loads(recortar_json(texto))
        show(f"{nome}: {type(dado).__name__} com chaves {list(dado)[:3]}")
    except json.JSONDecodeError as ex:
        show(f"{nome}: JSONDecodeError ({ex.msg})")

cerca ```json: dict com chaves ['prato', 'porcoes', 'ingredientes']
texto depois com chaves: JSONDecodeError (Extra data)
texto antes com chaves: JSONDecodeError (Expecting property name enclosed in double quotes)
lista solta: dict com chaves ['nome', 'quantidade', 'unidade']


O recorte transformou a lista solta no seu primeiro item — um `dict` — **sem erro**. Em vez de recortar, o `extrair_json`
abaixo **decodifica**: tenta ler um JSON a partir de cada `{` ou `[` (`json.JSONDecoder().raw_decode`) e devolve o
primeiro que é JSON de verdade, sem nada em volta.

In [12]:
def extrair_json(texto):
    """Primeiro JSON (objeto ou lista) decodificável no texto, como texto, sem nada em volta. O que não é texto passa."""
    if not isinstance(texto, str):
        return texto
    decoder = json.JSONDecoder()
    for inicio, caractere in enumerate(texto):
        if caractere in "{[":
            try:
                _, fim = decoder.raw_decode(texto, inicio)
            except json.JSONDecodeError:
                continue
            return texto[inicio:fim]
    raise ValueError("nenhum JSON decodificável na resposta")


for nome, texto in respostas.items():
    show(f"{nome}: {type(json.loads(extrair_json(texto))).__name__}")

cerca ```json: dict
texto depois com chaves: dict
texto antes com chaves: dict
lista solta: list


**A aplicação não toca em texto.** Ainda sobra um problema de desenho: espalhar `extrair_json` + `model_validate_json` pela
aplicação deixa **texto circulando como se fosse dado**. O Pydantic resolve com um **tipo de borda**: `Json[T]` recebe
texto JSON e valida contra `T`, e o `BeforeValidator` limpa antes o que vem em volta. Texto entra, objeto validado sai — e
esse é o único lugar onde existe texto:

In [13]:
from typing import Annotated

from pydantic import BeforeValidator, Json, TypeAdapter

SaidaListaDeCompras = TypeAdapter(Annotated[Json[ListaDeCompras], BeforeValidator(extrair_json)])

lista_da_borda = SaidaListaDeCompras.validate_python(respostas["texto depois com chaves"])
show(f"type(lista_da_borda).__name__: {type(lista_da_borda).__name__}")
show(f"lista_da_borda.prato: {lista_da_borda.prato}")

invalidas = {
    "sem JSON": "Desculpe, não consegui gerar a lista.",
    "quantidade zero": '{"prato": "x", "porcoes": 1, "ingredientes": [{"nome": "sal", "quantidade": 0, "unidade": "g"}]}',
}
for nome, texto in invalidas.items():
    try:
        SaidaListaDeCompras.validate_python(texto)
    except ValidationError as ex:
        erro = ex.errors()[0]
        show(f"{nome}: {erro['type']} em {erro['loc']}")

type(lista_da_borda).__name__: ListaDeCompras
lista_da_borda.prato: Pizza margherita
sem JSON: value_error em ()
quantidade zero: greater_than em ('ingredientes', 0, 'quantidade')


Os dois problemas chegam do mesmo jeito, como `ValidationError` com o lugar exato — é isso que volta ao modelo no retry do
[05](../02_intermediario/05_regras_e_retry.ipynb). Quando o endpoint suporta, a **saída nativa** (2.2) é ainda melhor: o SDK
entrega o objeto e não sobra texto nenhum. A partir do 03, toda resposta do modelo passa por um tipo de borda; no
intermediário, ele vira `saida_llm(Modelo)` no [`contrato.py`](../02_intermediario/contrato.py).

**Resumo:** o contrato garante a **forma** (campos e tipos). O vocabulário de cada campo — quais unidades, quais categorias — ainda
está solto. ➡️ [03 — enums e guardrails](03_enums_e_guardrails.ipynb)